In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

/Users/martindufour/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Function 7
print('Function 7')
func7_inputs = np.load('./initial_data/function_7/initial_inputs.npy')
print(func7_inputs)

func7_outputs = np.load('./initial_data/function_7/initial_outputs.npy')
print(func7_outputs)
print('/n')

Function 7
[[0.27262382 0.32449536 0.89710881 0.83295115 0.15406269 0.79586362]
 [0.54300258 0.9246939  0.34156746 0.64648585 0.71844033 0.34313266]
 [0.09083225 0.66152938 0.06593091 0.25857701 0.96345285 0.6402654 ]
 [0.11886697 0.61505494 0.90581639 0.8553003  0.41363143 0.58523563]
 [0.63021764 0.8380969  0.68001305 0.73189509 0.52673671 0.34842921]
 [0.76491917 0.25588292 0.60908422 0.21807904 0.32294277 0.09579366]
 [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
 [0.19525188 0.07922665 0.55458046 0.17056682 0.01494418 0.10703171]
 [0.64230298 0.83687455 0.02179269 0.10148801 0.68307083 0.6924164 ]
 [0.78994255 0.19554501 0.57562333 0.07365919 0.25904917 0.05109986]
 [0.52849733 0.45742436 0.36009569 0.36204551 0.81689098 0.63747637]
 [0.72261522 0.01181284 0.06364591 0.16517311 0.07924415 0.35995166]
 [0.07566492 0.33450212 0.13273274 0.60831236 0.91838592 0.82233079]
 [0.94245084 0.37743962 0.48612233 0.22879108 0.08263175 0.71195755]
 [0.14864702 0.03394336

In [3]:
# Week 1 Input and Output Data
week_1_inputs = [ np.array([0.155793, 0.528435]), np.array([0.224164, 0.812385]), np.array([0.830919, 0.158523, 0.528406]), np.array([0.912533, 0.052672, 0.771239, 0.219812]), np.array([0.234189, 0.83648 , 0.884484, 0.873516]), np.array([0.490808, 0.618683, 0.277824, 0.900494, 0.106596]), np.array([0.067896, 0.486672, 0.255422, 0.215118, 0.427428, 0.72097 ]), np.array([0.061447, 0.062956, 0.029929, 0.036786, 0.407935, 0.795055, 0.496307,0.888085]) ]
week_1_outputs = [np.float64(1.5311489892413605e-58), np.float64(0.04614596805685454), np.float64(-0.04591165123945737), np.float64(-24.387232512869755), np.float64(1049.4420694211206), np.float64(-0.849252655626155), np.float64(1.3793294734939503), np.float64(9.598780741169)]

week_2_inputs = [np.array([0.946399, 0.18731 ]), np.array([0.712637, 0.921564]), np.array([0.765104, 0.052672, 0.438597]), np.array([0.380965, 0.771701, 0.089479, 0.536968]), np.array([0.219189, 0.85148 , 0.874484, 0.883516]), np.array([0.114755, 0.697421, 0.354179, 0.887624, 0.589139]), np.array([0.070896, 0.484672, 0.259422, 0.216118, 0.427428, 0.72297 ]), np.array([0.062447, 0.061956, 0.030929, 0.035786, 0.408935, 0.794055, 0.497307, 0.887085])]
week_2_outputs = [np.float64(2.338449488843798e-206), np.float64(0.5728372778652475), np.float64(-0.10235480701941752), np.float64(-13.368584815829887), np.float64(1109.9883069580462), np.float64(-1.492909868264592), np.float64(1.3944037721068683), np.float64(9.598978827169)]

week_3_inputs = [np.array([0.583738, 0.706798]), np.array([0.699637, 0.928564]), np.array([0.123457, 0.876543, 0.5     ]), np.array([0.230785, 0.914568, 0.102938, 0.657322]), np.array([0.214189, 0.85648 , 0.869484, 0.888516]), np.array([0.051235, 0.987654, 0.43211 , 0.123457, 0.765432]), np.array([0.071896, 0.483672, 0.261422, 0.217118, 0.426428, 0.72497 ]), np.array([0.063447, 0.060956, 0.031929, 0.034786, 0.409935, 0.793055, 0.498307, 0.886085])]
week_3_outputs = [np.float64(9.817718646044271e-07), np.float64(0.49978778689465564), np.float64(-0.04054152149606349), np.float64(-21.112987453792396), np.float64(1132.5255136709882), np.float64(-2.4679805862566795), np.float64(1.4059619293543582), np.float64(9.599155713169)]

week_4_inputs = [np.array([0.867072, 0.913241]), np.array([0.83604 , 0.696071]), np.array([0.447658, 0.395195, 0.505344]), np.array([0.422706, 0.385497, 0.37529 , 0.410833]), np.array([0.157706, 0.912326, 0.830158, 0.925715]), np.array([0.275401, 0.      , 0.568079, 1.      , 0.121326]), np.array([0.124679, 0.389027, 0.389012, 0.23686 , 0.379036, 0.802247]), np.array([0.077447, 0.21029 , 0.114032, 0.159535, 0.695924, 0.531316,0.178973, 0.57168 ])]
week_4_outputs = [np.float64(-1.0508862613282513e-96), np.float64(0.20657541488475104), np.float64(-0.03229682155733878), np.float64(0.4964978124935766), np.float64(1445.380906735266), np.float64(-0.8152779914672599), np.float64(2.0217812896063525), np.float64(9.987130922543)]

week_5_inputs = [np.array([0.065052, 0.948886]), np.array([0.388677, 0.271349]), np.array([1.      , 0.136717, 0.850593]), np.array([0.908266, 0.239562, 0.144895, 0.489453]), np.array([0.115614, 0.955641, 0.820859, 0.938174]), np.array([0.568442, 0.      , 1.      , 1.      , 1.      ]), np.array([0.134015, 0.028783, 0.755137, 0.62031 , 0.70408 , 0.212964]), np.array([0.127008, 0.292876, 0.06967 , 0.277582, 0.553407, 0.547258,0.220835, 0.443146])]
week_5_outputs = [np.float64(2.0262778967114778e-283), np.float64(0.016418658339648333), np.float64(-0.054388754089278846), np.float64(-17.161465002411145), np.float64(1779.8600577462366), np.float64(-1.9906490554141107), np.float64(0.052467603616080494), np.float64(9.9149654065019)]

week_6_inputs = [np.array([0.17701 , 0.088703]), np.array([0.593592, 0.679102]), np.array([0.7536, 1.    , 1.    ]), np.array([0.370348, 0.379959, 0.430216, 0.444351]), np.array([0.169493, 0.556801, 0.936155, 0.69603 ]), np.array([0.366464, 0.316099, 1.      , 1.      , 0.      ]), np.array([0.123574, 0.270452, 0.482301, 0.208163, 0.330398, 0.872733]), np.array([0.3191  , 0.828915, 0.037008, 0.59627 , 0.230009, 0.120567,0.076953, 0.696289])]
week_6_outputs = [np.float64(-2.3725238219366144e-119), np.float64(0.06836721478932847), np.float64(-0.48310415434111403), np.float64(0.18076540708623456), np.float64(282.83820524691396), np.float64(-0.8214281898088153), np.float64(2.075605759888563), np.float64(8.8803427965834)]

week_7_inputs = [np.array([0.065052, 0.948886]), np.array([0.140924, 0.802197]), np.array([1., 1., 0.]), np.array([0.32078 , 0.186519, 0.040775, 0.590893]), np.array([0.548734, 0.691895, 0.651961, 0.224269]), np.array([0.368433, 0.      , 1.      , 1.      , 0.428022]), np.array([0.139689, 0.317433, 0.463608, 0.250431, 0.325485, 0.810756]), np.array([0.      , 0.202823, 0.231703, 0.      , 1.      , 1.      ,
       0.288474, 1.      ])]
week_7_outputs = [np.float64(2.0262778967114778e-283), np.float64(-0.10826299524356352), np.float64(-0.16354530625442043), np.float64(-11.58523458824008), np.float64(1.9931553503870212), np.float64(-1.2796687884296385), np.float64(2.462201676843452), np.float64(9.622023932692)]

week_8_inputs = [np.array([0.000788, 0.033717]), np.array([0.914607, 0.789979]), np.array([0.317253, 0.002183, 0.963506]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.078217, 0.973099, 0.868006, 0.933352]), np.array([0.426158, 0.348959, 0.616644, 0.692851, 0.024814]), np.array([0.269889, 0.346609, 0.467584, 0.256632, 0.302654, 0.800092]), np.array([0.066269, 0.029193, 0.143059, 0.209133, 0.840619, 0.604919,
       0.230297, 0.701468])]
week_8_outputs = [np.float64(1.5608341712501477e-228), np.float64(0.0347797753016137), np.float64(-0.3756702789549372), np.float64(-33.661790988299735), np.float64(2151.3700669834334), np.float64(-0.23000336822278494), np.float64(2.4516632535923746), np.float64(9.9644234438351)]

week_9_inputs = [np.array([0.000924, 0.003116]), np.array([0.914607, 0.789979]), np.array([0.047574, 0.998325, 0.999125]), np.array([0.008334, 0.234163, 0.946857, 0.993453]), np.array([0.080698, 0.971918, 0.842327, 0.954471]), np.array([0.942348, 0.037756, 0.105925, 0.010348, 0.997595]), np.array([0.090198, 0.680559, 0.851748, 0.080076, 0.298474, 0.701193]), np.array([0.95983 , 0.002073, 0.011779, 0.227207, 0.571611, 0.031204,
       0.24468 , 0.055697])]
week_9_outputs = [np.float64(7.25285761175276e-246), np.float64(0.05670257386671321), np.float64(-0.48158498276260003), np.float64(-33.661790988299735), np.float64(2156.522419821577), np.float64(-3.026993482898997), np.float64(0.6117443854593981), np.float64(8.1721431718416)]

week_10_inputs = [np.array([0.999878, 0.001995]), np.array([0.41611 , 0.666998]), np.array([0.001378, 0.999625, 0.642937]), np.array([0.027746, 0.654937, 0.999652, 0.024674]), np.array([0.109191, 0.976767, 0.866838, 0.873854]), np.array([0.944032, 0.013284, 0.976966, 0.023315, 0.997903]), np.array([0.167912, 0.257921, 0.480203, 0.472529, 0.254657, 0.778833]), np.array([0.116951, 0.002708, 0.212486, 0.121597, 0.986678, 0.488014,
       0.152731, 0.379341])]
week_10_outputs = [np.float64(0.0), np.float64(0.1625184332362701), np.float64(-0.13457392031599885), np.float64(-30.133085064011215), np.float64(1769.6901405375768), np.float64(-2.7724961974553874), np.float64(1.9208154093840464), np.float64(9.9296060847489)]

week_11_inputs = [np.array([0.37454 , 0.950714]), np.array([0.41611 , 0.666998]), np.array([0.997078, 0.475121, 0.651523]), np.array([0.00109 , 0.902069, 0.972212, 0.16754 ]), np.array([0.019324, 0.941663, 0.833048, 0.950636]), np.array([0.973398, 0.011482, 0.130865, 0.931063, 0.997256]), np.array([0.215391, 0.288991, 0.602008, 0.322698, 0.265768, 0.811541]), np.array([0.16109 , 0.049964, 0.210829, 0.130137, 0.965223, 0.408414,
       0.113403, 0.289162])]
week_11_outputs = [np.float64(-1.560646704467778e-117), np.float64(-0.1334547156009971), np.float64(-0.10208280924057045), np.float64(-31.74483921038956), np.float64(1827.9676185989063), np.float64(-2.432448557520746), np.float64(2.4819294367786457), np.float64(9.9158368797091)]

week_12_inputs = [np.array([0.001   , 0.999576]), np.array([0.45253, 0.6577 ]), np.array([0.998539, 0.483668, 0.657653]), np.array([0.018818, 0.998904, 0.999658, 0.998016]), np.array([0.064807, 0.978215, 0.856404, 0.971059]), np.array([0.898391, 0.002198, 0.019804, 0.999554, 0.917075]), np.array([0.211376, 0.335246, 0.510659, 0.146838, 0.289515, 0.653795]), np.array([4.18000e-04, 1.05120e-02, 1.95045e-01, 2.11750e-01, 7.88533e-01,
       9.00374e-01, 2.41123e-01, 3.34541e-01])]
week_12_outputs = [np.float64(0.0), np.float64(0.3536071649472596), np.float64(-0.09423686717649356), np.float64(-47.62748092409266), np.float64(2449.5648412319215), np.float64(-2.500439911687528), np.float64(2.549261085159255), np.float64(9.7734101935864)]

In [4]:
# Function 7
print('Function 7')
# Load inputs from previous run
# Loads initial data
week_0_func7_inputs = np.load('./initial_data/function_7/initial_inputs.npy')
week_0_func7_outputs = np.load('./initial_data/function_7/initial_outputs.npy')

print(f'Shape of initial input data: {week_0_func7_inputs.shape}')
print(f'Shape of initial output data: {week_0_func7_outputs.shape}')

print(f'Week 1 inputs: {week_1_inputs[6]}')
print(f'Week 2 inputs: {week_2_inputs[6]}')
print(f'Week 3 inputs: {week_3_inputs[6]}')
print(f'Week 4 inputs: {week_4_inputs[6]}')
print(f'Week 5 inputs: {week_5_inputs[6]}')
print(f'Week 6 inputs: {week_6_inputs[6]}')
print(f'Week 7 inputs: {week_7_inputs[6]}')
print(f'Week 8 inputs: {week_8_inputs[6]}')
print(f'Week 9 inputs: {week_9_inputs[6]}')
print(f'Week 10 inputs: {week_10_inputs[6]}')
print(f'Week 11 inputs: {week_11_inputs[6]}')
print(f'Week 12 inputs: {week_12_inputs[6]}')

combined_func7_inputs = np.vstack([
    week_0_func7_inputs,
    week_1_inputs[6],
    week_2_inputs[6],
    week_3_inputs[6],
    week_4_inputs[6],
    week_5_inputs[6],
    week_6_inputs[6],
    week_7_inputs[6],
    week_8_inputs[6],
    week_9_inputs[6],
    week_10_inputs[6],
    week_11_inputs[6],
    week_12_inputs[6]
])
print(f'Number of input data points: {len(combined_func7_inputs)}')
print('Combined input data')
print(combined_func7_inputs)

# Load outputs from previous run
week_func7_output = week_1_outputs[6]
combined_func7_outputs = np.concatenate([
    week_0_func7_outputs,
    [week_1_outputs[6]],
    [week_2_outputs[6]],
    [week_3_outputs[6]],
    [week_4_outputs[6]],
    [week_5_outputs[6]],
    [week_6_outputs[6]],
    [week_7_outputs[6]],
    [week_8_outputs[6]],
    [week_9_outputs[6]],
    [week_10_outputs[6]],
    [week_11_outputs[6]],
    [week_12_outputs[6]]
])
print(f'Number of output data points: {len(combined_func7_outputs)}')
print('Combined output data')
print(combined_func7_outputs)

Function 7
Shape of initial input data: (30, 6)
Shape of initial output data: (30,)
Week 1 inputs: [0.067896 0.486672 0.255422 0.215118 0.427428 0.72097 ]
Week 2 inputs: [0.070896 0.484672 0.259422 0.216118 0.427428 0.72297 ]
Week 3 inputs: [0.071896 0.483672 0.261422 0.217118 0.426428 0.72497 ]
Week 4 inputs: [0.124679 0.389027 0.389012 0.23686  0.379036 0.802247]
Week 5 inputs: [0.134015 0.028783 0.755137 0.62031  0.70408  0.212964]
Week 6 inputs: [0.123574 0.270452 0.482301 0.208163 0.330398 0.872733]
Week 7 inputs: [0.139689 0.317433 0.463608 0.250431 0.325485 0.810756]
Week 8 inputs: [0.269889 0.346609 0.467584 0.256632 0.302654 0.800092]
Week 9 inputs: [0.090198 0.680559 0.851748 0.080076 0.298474 0.701193]
Week 10 inputs: [0.167912 0.257921 0.480203 0.472529 0.254657 0.778833]
Week 11 inputs: [0.215391 0.288991 0.602008 0.322698 0.265768 0.811541]
Week 12 inputs: [0.211376 0.335246 0.510659 0.146838 0.289515 0.653795]
Number of input data points: 42
Combined input data
[[0.27262

In [5]:
### ====== OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 7 ======
# Import the OptunaBayesianOptimizer class
import sys
sys.path.insert(0, './bayesian_optimization_challenge-md')
from bo_optuna import OptunaBayesianOptimizer

# Create optimizer instance with initial Function 7 data
print("=" * 60)
print("OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 7")
print("=" * 60)

X_func7_initial = week_0_func7_inputs
y_func7_initial = week_0_func7_outputs
bounds_func7 = [(0, 1)] * X_func7_initial.shape[1]

optimizer_func7 = OptunaBayesianOptimizer(
    X_initial=X_func7_initial,
    y_initial=y_func7_initial,
    bounds=bounds_func7,
    optimize_hp=True,  # Enable hyperparameter tuning
    random_state=42,
    acquisition="ei"
)

print(f"\nInitial training data shape: X={optimizer_func7.X_train.shape}, y={optimizer_func7.y_train.shape}")
print(f"Initial best observation: {optimizer_func7.get_best_observation()[1]:.6e}")


OPTUNA-BASED BAYESIAN OPTIMIZATION FOR FUNCTION 7

Initial training data shape: X=(30, 6), y=(30,)
Initial best observation: 1.364968e+00


In [6]:
# Run Optuna-based BO for 8 weeks (8 iterations)
print("\nRunning Optuna-based BO for 8 iterations...")
print("-" * 60)

# Get all weekly data
weekly_data = [
    (week_1_inputs[6], week_1_outputs[6]),
    (week_2_inputs[6], week_2_outputs[6]),
    (week_3_inputs[6], week_3_outputs[6]),
    (week_4_inputs[6], week_4_outputs[6]),
    (week_5_inputs[6], week_5_outputs[6]),
    (week_6_inputs[6], week_6_outputs[6]),
    (week_7_inputs[6], week_7_outputs[6]),
    (week_8_inputs[6], week_8_outputs[6]),
    (week_9_inputs[6], week_9_outputs[6]),
    (week_10_inputs[6], week_10_outputs[6]),
    (week_11_inputs[6], week_11_outputs[6]),
    (week_12_inputs[6], week_12_outputs[6])
]

optuna_proposals_func7 = []
manual_best_func7 = week_0_func7_outputs.max()
optuna_best_func7 = y_func7_initial.max()

for week, (x_actual, y_actual) in enumerate(weekly_data, start=1):
    print(f"\nWeek {week}:")
    print(f"  Actual observation: y = {y_actual:.6e}")
    
    # Get Optuna proposal
    proposals = optimizer_func7.optimize(
        n_iterations=1,
        optimize_hp_every=1 if week % 2 == 0 else 0,  # Tune HP every other week
        optimize_hp_n_trials=30,
        acq_n_trials=100,
        verbose=True
    )
    
    x_proposed = proposals[0]
    optuna_proposals_func7.append(x_proposed)
    
    # Update optimizer with actual observation
    optimizer_func7.update(x_actual, y_actual)
    
    # Track best values
    manual_best_func7 = max(manual_best_func7, y_actual)
    optuna_best_func7 = max(optuna_best_func7, y_actual)
    
    print(f"  Best so far (Optuna): {optuna_best_func7:.6e}")

print("\n" + "=" * 60)
print("OPTUNA-BASED BO COMPLETED FOR FUNCTION 7")
print("=" * 60)


[I 2026-05-04 06:40:59,180] A new study created in memory with name: no-name-9663c71a-a9d1-4aab-b1a0-97a42897178b
[I 2026-05-04 06:40:59,209] Trial 0 finished with value: 1.8716555139017548e-06 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162, 'x2': 0.7319939418114051, 'x3': 0.5986584841970366, 'x4': 0.15601864044243652, 'x5': 0.15599452033620265}. Best is trial 0 with value: 1.8716555139017548e-06.
[I 2026-05-04 06:40:59,211] Trial 1 finished with value: 1.620172695143096e-06 and parameters: {'x0': 0.05808361216819946, 'x1': 0.8661761457749352, 'x2': 0.6011150117432088, 'x3': 0.7080725777960455, 'x4': 0.020584494295802447, 'x5': 0.9699098521619943}. Best is trial 0 with value: 1.8716555139017548e-06.
[I 2026-05-04 06:40:59,213] Trial 2 finished with value: 9.621200104962908e-09 and parameters: {'x0': 0.8324426408004217, 'x1': 0.21233911067827616, 'x2': 0.18182496720710062, 'x3': 0.18340450985343382, 'x4': 0.3042422429595377, 'x5': 0.5247564316322378}. Best is trial 


Running Optuna-based BO for 8 iterations...
------------------------------------------------------------

Week 1:
  Actual observation: y = 1.379329e+00


[I 2026-05-04 06:40:59,284] Trial 13 finished with value: 2.046525234522523e-05 and parameters: {'x0': 0.2158360407128856, 'x1': 0.6803951215720511, 'x2': 0.4098750341706901, 'x3': 0.33710262029031846, 'x4': 0.7667236863442447, 'x5': 0.04535002119833897}. Best is trial 12 with value: 2.178356727245513e-05.
[I 2026-05-04 06:40:59,296] Trial 14 finished with value: 8.22846556451854e-06 and parameters: {'x0': 0.21526662688351697, 'x1': 0.47428268266554097, 'x2': 0.422205883108772, 'x3': 0.34279928826631634, 'x4': 0.8469961233886654, 'x5': 0.0016222719066337453}. Best is trial 12 with value: 2.178356727245513e-05.
[I 2026-05-04 06:40:59,310] Trial 15 finished with value: 1.302817916381803e-05 and parameters: {'x0': 0.18902137821633555, 'x1': 0.4885751247830854, 'x2': 0.47664044263131045, 'x3': 0.32716743009924104, 'x4': 0.7640242188605716, 'x5': 0.1779635150470358}. Best is trial 12 with value: 2.178356727245513e-05.
[I 2026-05-04 06:40:59,323] Trial 16 finished with value: 0.0004555730018

[Iteration 0] Proposed: [0.09821866 0.37710806 0.2739816  0.10758208 0.41969893 0.64072641], EI: 0.030273
  Best so far (Optuna): 1.379329e+00

Week 2:
  Actual observation: y = 1.394404e+00


[I 2026-05-04 06:41:00,683] Trial 13 finished with value: 2.183096668910428e-05 and parameters: {'x0': 0.20833468440279257, 'x1': 0.4664285333625645, 'x2': 0.4206460428766417, 'x3': 0.012699250231953603, 'x4': 0.7716641432569382, 'x5': 0.22931425055978924}. Best is trial 11 with value: 6.519828694896961e-05.
[I 2026-05-04 06:41:00,695] Trial 14 finished with value: 8.608290877522436e-06 and parameters: {'x0': 0.22778072170105215, 'x1': 0.47428268266554097, 'x2': 0.4540930649750778, 'x3': 0.009998053671139043, 'x4': 0.8469961233886654, 'x5': 0.2607723829332806}. Best is trial 11 with value: 6.519828694896961e-05.
[I 2026-05-04 06:41:00,709] Trial 15 finished with value: 3.6353580136409448e-06 and parameters: {'x0': 0.18384970480099358, 'x1': 0.6071250671236179, 'x2': 0.48293557141275156, 'x3': 0.2924562576122224, 'x4': 0.7670095005998434, 'x5': 0.6855046081289344}. Best is trial 11 with value: 6.519828694896961e-05.
[I 2026-05-04 06:41:00,722] Trial 16 finished with value: 2.64386376660

...

[Iteration 0] Proposed: [0.21539089 0.28899127 0.60200816 0.32269849 0.26576819 0.81154122], EI: 0.156705
  Best so far (Optuna): 2.462202e+00

Week 11:
  Actual observation: y = 2.481929e+00


[I 2026-05-04 06:41:14,366] A new study created in memory with name: no-name-9e513a63-0174-40d1-8894-d93ea69fb5f6
[I 2026-05-04 06:41:14,368] Trial 0 finished with value: 5.123033182004018e-08 and parameters: {'x0': 0.3745401188473625, 'x1': 0.9507143064099162, 'x2': 0.7319939418114051, 'x3': 0.5986584841970366, 'x4': 0.15601864044243652, 'x5': 0.15599452033620265}. Best is trial 0 with value: 5.123033182004018e-08.
[I 2026-05-04 06:41:14,370] Trial 1 finished with value: 4.6890670498627504e-08 and parameters: {'x0': 0.05808361216819946, 'x1': 0.8661761457749352, 'x2': 0.6011150117432088, 'x3': 0.7080725777960455, 'x4': 0.020584494295802447, 'x5': 0.9699098521619943}. Best is trial 0 with value: 5.123033182004018e-08.
[I 2026-05-04 06:41:14,372] Trial 2 finished with value: 1.747421960815866e-08 and parameters: {'x0': 0.8324426408004217, 'x1': 0.21233911067827616, 'x2': 0.18182496720710062, 'x3': 0.18340450985343382, 'x4': 0.3042422429595377, 'x5': 0.5247564316322378}. Best is trial 0 

[Iteration 0] Proposed: [0.21137562 0.33524568 0.51065938 0.14683815 0.28951532 0.65379512], EI: 0.046830
  Best so far (Optuna): 2.481929e+00

Week 12:
  Actual observation: y = 2.549261e+00


[I 2026-05-04 06:41:15,872] Trial 4 finished with value: 5.136362804696187e-08 and parameters: {'x0': 0.45606998421703593, 'x1': 0.7851759613930136, 'x2': 0.19967378215835974, 'x3': 0.5142344384136116, 'x4': 0.5924145688620425, 'x5': 0.046450412719997725}. Best is trial 3 with value: 1.7134422728669725e-06.
[I 2026-05-04 06:41:15,875] Trial 5 finished with value: 3.09384703789259e-08 and parameters: {'x0': 0.6075448519014384, 'x1': 0.17052412368729153, 'x2': 0.06505159298527952, 'x3': 0.9488855372533332, 'x4': 0.9656320330745594, 'x5': 0.8083973481164611}. Best is trial 3 with value: 1.7134422728669725e-06.
[I 2026-05-04 06:41:15,878] Trial 6 finished with value: 6.9105253721341305e-09 and parameters: {'x0': 0.3046137691733707, 'x1': 0.09767211400638387, 'x2': 0.6842330265121569, 'x3': 0.4401524937396013, 'x4': 0.12203823484477883, 'x5': 0.4951769101112702}. Best is trial 3 with value: 1.7134422728669725e-06.
[I 2026-05-04 06:41:15,880] Trial 7 finished with value: 1.9415362071495836e-

[Iteration 0] Proposed: [0.17855064 0.33192854 0.54166136 0.30178952 0.25811467 0.64332105], EI: 0.115480
  Best so far (Optuna): 2.549261e+00

OPTUNA-BASED BO COMPLETED FOR FUNCTION 7
